# 02. Reconciliation

The hierarchy from the lower 48 to the subregion, three methods, coherence to the megawatt, and
the honest result by level. The dead end kept here: top down by historical proportions, which
made the leaves worse, as the table shows.

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
manifest = json.loads((ROOT / "results" / "manifest.json").read_text(encoding="utf-8"))
values = manifest["values"]
v = lambda key: values[key]["value"]
print("as of", manifest["as_of"], "with", len(values), "values")

## The summing matrix

Built from `data/hierarchy.csv` with a remainder node per authority that has subregions, so the
leaves sum to their authority without any series being rescaled.

In [ ]:
nodes = pl.read_csv(ROOT / "results" / "hierarchy" / "nodes.csv")
print(nodes.group_by("kind").len().sort("kind"))
from lookahead_hierarchy.summing import SummingMatrix
summing = SummingMatrix.from_table(nodes)
print("nodes", len(summing.nodes), "leaves", len(summing.leaves), "matrix", summing.matrix.shape)

## Coherence and accuracy by level

In [ ]:
coherence = json.loads((ROOT / "results" / "hierarchy" / "coherence.json").read_text())
print("coherence gaps in MW:", coherence["gaps_mw"])
scores = pl.read_parquet(ROOT / "results" / "hierarchy" / "scores_by_level.parquet")
print(scores.select("method", "level_name", "mape", "mape_lower", "mape_upper", "coverage_90").sort(["level", "method"]))

## The dead end: top down hurts the leaves

Splitting the top forecast by the validation year's proportions ignores everything the leaf
models know, and the remainder nodes make the proportions noisy. It is kept in the table and on
the page because the morning meeting reads the leaves too.

In [ ]:
base = scores.filter(pl.col("method") == "base").sort("level")
for method in ("bottom_up", "top_down", "mint"):
    m = scores.filter(pl.col("method") == method).sort("level")
    change = (m["mape"] - base["mape"]).to_list()
    print(method, "change in MAPE by level (negative helps):", [round(c, 4) for c in change])
print("helped:", v("hierarchy.helped_list"))
print("hurt:", v("hierarchy.hurt_list"))

## Against the truth, on the simulator

On the simulator the reconciliation is scored against the true demand rather than the observed
actuals, by condition and seed.

In [ ]:
summary = pl.read_parquet(ROOT / "results" / "recovery" / "summary.parquet")
print(summary.filter(pl.col("figure").str.starts_with("reconciliation.mint_gain")).select("condition", "figure", "mean", "lower", "upper"))